# NorthStar Urban Mobility — Section 1: SQL in R

**Module:** Databases and Analytics  
**Case Study:** NorthStar Urban Mobility and Logistics

## Aim of this notebook
Use SQL queries within R (via the `sqldf` package) to investigate three core problems flagged by NorthStar's executive team:

1. **The 'completed-but-failed' paradox** — orders marked successful in one system while customers complain in another.
2. **The route-override mystery** — drivers manually overriding planned routes; is this driving failure?
3. **The hidden cost story** — where compensation, delays, and incidents really concentrate.

We use R + SQL for structured, set-based querying across the operational tables, then R's `dplyr`/`ggplot2` for transformation and visualisation.

---

## 1. Switch the Colab runtime to R

This notebook **must** run on an R runtime, not Python.  
In Colab: **Runtime → Change runtime type → Runtime type: R → Save**.

Verify with the cell below.

In [ ]:
R.version.string
cat('Working dir:', getwd(), '\n')

## 2. Install and load packages

- `sqldf`  — run SQL on R data frames
- `dplyr` — data manipulation grammar
- `ggplot2` — visualisation
- `lubridate` — date/time handling
- `readr` — fast CSV import
- `tidyr` — reshaping

In [ ]:
pkgs <- c('sqldf', 'dplyr', 'ggplot2', 'lubridate', 'readr', 'tidyr', 'scales')
to_install <- pkgs[!pkgs %in% installed.packages()[,'Package']]
if (length(to_install) > 0) install.packages(to_install)

suppressPackageStartupMessages({
  library(sqldf)
  library(dplyr)
  library(ggplot2)
  library(lubridate)
  library(readr)
  library(tidyr)
  library(scales)
})

options(repr.plot.width = 9, repr.plot.height = 5)
options(width = 200)   # wider console so SQL result tables don't wrap
cat('All packages loaded.\n')

## 3. Import the dataset from GitHub

**How to import the dataset to GitHub** *(written explanation for the report):*
1. Create a new public repository on GitHub (e.g. `northstar-databases-analytics`).
2. Inside the repo, create a folder `data/`.
3. Upload all 9 CSV files via the GitHub web UI (`Add file → Upload files`) or `git add data/*.csv && git push`.
4. For each file, click the file → click **Raw** → copy the URL. The raw URL pattern is:  
   `https://raw.githubusercontent.com/<USER>/<REPO>/main/data/<FILE>.csv`
5. Read the raw URL directly with `readr::read_csv()` — no download step needed.

**Replace `GITHUB_USER` and `REPO` below with your own values** before running.

In [ ]:
# >>> EDIT THESE TWO LINES <<<
GITHUB_USER <- 'YOUR_GITHUB_USERNAME'
REPO        <- 'northstar-databases-analytics'
BRANCH      <- 'main'

base_url <- sprintf('https://raw.githubusercontent.com/%s/%s/%s/data/', GITHUB_USER, REPO, BRANCH)

# --- Fallback: if you haven't pushed to GitHub yet, upload the CSVs to Colab
# (left sidebar → Files → upload) and set USE_LOCAL <- TRUE.
USE_LOCAL <- TRUE   # set to FALSE once your GitHub URLs are live

load_csv <- function(name) {
  path <- if (USE_LOCAL) paste0(name, '.csv') else paste0(base_url, name, '.csv')
  read_csv(path, show_col_types = FALSE)
}

hubs       <- load_csv('hubs')
customers  <- load_csv('customers')
drivers    <- load_csv('drivers')
vehicles   <- load_csv('vehicles')
orders     <- load_csv('orders')
deliveries <- load_csv('deliveries')
incidents  <- load_csv('incidents')
complaints <- load_csv('complaints')
app_events <- load_csv('app_events')

cat('Rows loaded:\n')
cat(sprintf('  hubs=%d, customers=%d, drivers=%d, vehicles=%d\n',
            nrow(hubs), nrow(customers), nrow(drivers), nrow(vehicles)))
cat(sprintf('  orders=%d, deliveries=%d, incidents=%d, complaints=%d, app_events=%d\n',
            nrow(orders), nrow(deliveries), nrow(incidents), nrow(complaints), nrow(app_events)))

## 4. Quick data quality scan

The case-study brief warns of inconsistent values. Let's confirm before any analysis.

In [ ]:
cat('--- Zone values across tables (note the casing inconsistency) ---\n')
cat('customers$home_zone :', sort(unique(customers$home_zone)), '\n')
cat('drivers$base_zone   :', sort(unique(drivers$base_zone)), '\n')
cat('orders$pickup_zone  :', sort(unique(orders$pickup_zone)), '\n')
cat('app_events$zone_context:', sort(unique(app_events$zone_context)), '\n\n')

cat('--- Missing-value counts ---\n')
for (nm in c('customers','drivers','vehicles','orders','deliveries','incidents','complaints','app_events')) {
  df <- get(nm)
  na_cols <- colSums(is.na(df))
  na_cols <- na_cols[na_cols > 0]
  if (length(na_cols)) cat(sprintf('%-12s : %s\n', nm, paste(names(na_cols), na_cols, sep='=', collapse=', ')))
}

**Finding (already a deliverable for the report):** the `zone` column appears in four tables with *inconsistent casing*: `Airport / AIRPORT / airport / Airport_Hub` etc. This is the data-fragmentation symptom the technology director warned about. Before any cross-table SQL, we must normalise these.

In [ ]:
# Standardise zone values to Title Case across all tables
tidy_zone <- function(x) {
  x <- tools::toTitleCase(tolower(trimws(x)))
  x[x == 'Airport_Hub'] <- 'Airport'
  x[x == 'Ctr']         <- 'Central'   # abbreviation found in raw orders data
  x
}
customers$home_zone     <- tidy_zone(customers$home_zone)
drivers$base_zone       <- tidy_zone(drivers$base_zone)
vehicles$assigned_zone  <- tidy_zone(vehicles$assigned_zone)
orders$pickup_zone      <- tidy_zone(orders$pickup_zone)
orders$dropoff_zone     <- tidy_zone(orders$dropoff_zone)
app_events$zone_context <- tidy_zone(app_events$zone_context)
hubs$zone               <- tidy_zone(hubs$zone)

cat('Zones now consistent across tables:\n')
cat('  customers:', sort(unique(customers$home_zone)), '\n')
cat('  hubs     :', sort(unique(hubs$zone)), '\n')

---
## 5. SQL operations: SELECT / INSERT / UPDATE / DELETE

`sqldf` lets us run SQL directly against R data frames as if they were tables.  
We work on a copy (`orders_sql`) so the canonical `orders` data frame is preserved.

In [ ]:
orders_sql <- as.data.frame(orders)

# ---------- SELECT ----------
q_select <- "
  SELECT order_id, service_type, priority_level, order_value, pickup_zone, dropoff_zone
  FROM orders_sql
  WHERE priority_level IN ('High','Critical')
    AND order_value > 100
  ORDER BY order_value DESC
  LIMIT 8
"
cat('--- SELECT: high-priority high-value orders ---\n')
sqldf(q_select)

In [ ]:
# ---------- INSERT ----------
# Add a new test order (e.g. a critical medical service ride)
q_insert <- "
  INSERT INTO orders_sql
    (order_id, customer_id, service_type, order_created_at,
     promised_window_hours, pickup_zone, dropoff_zone,
     priority_level, order_value, booking_channel, special_handling_flag)
  VALUES
    ('O99999','C0001','Medical','2025-12-31 23:59:00',
     2,'Central','Airport','Critical',299.99,'App',1)
"
orders_sql <- sqldf(c(q_insert, 'SELECT * FROM orders_sql'))
cat('Row count after INSERT:', nrow(orders_sql), '\n')
sqldf("SELECT * FROM orders_sql WHERE order_id='O99999'")

In [ ]:
# ---------- UPDATE ----------
# Bump priority of the test order from Critical -> High after re-triage
q_update <- "UPDATE orders_sql SET priority_level='High', order_value=order_value*0.9
             WHERE order_id='O99999'"
orders_sql <- sqldf(c(q_update, 'SELECT * FROM orders_sql'))
sqldf("SELECT order_id, priority_level, order_value FROM orders_sql WHERE order_id='O99999'")

In [ ]:
# ---------- DELETE ----------
q_delete <- "DELETE FROM orders_sql WHERE order_id='O99999'"
orders_sql <- sqldf(c(q_delete, 'SELECT * FROM orders_sql'))
cat('Row count after DELETE:', nrow(orders_sql),
    '(original was', nrow(orders), ')\n')

---
## 6. Mathematical, aggregate and arithmetic functions

Standard SQL aggregates (`AVG`, `COUNT`, `SUM`, `MIN`, `MAX`) plus arithmetic in `SELECT`.

In [ ]:
# Aggregates by service type — basic sanity baseline
q_agg <- "
  SELECT
    service_type,
    COUNT(*)                              AS orders_count,
    ROUND(AVG(order_value),2)             AS avg_value,
    ROUND(SUM(order_value),2)             AS total_value,
    MIN(order_value)                      AS min_value,
    MAX(order_value)                      AS max_value,
    ROUND(AVG(promised_window_hours),2)   AS avg_window_h
  FROM orders
  GROUP BY service_type
  ORDER BY total_value DESC
"
agg_result <- sqldf(q_agg)
agg_result

### 6.1 SQL query optimisation note

Two design choices in the SQL above already reflect optimisation thinking, and we can verify with `EXPLAIN QUERY PLAN`:

1. **Aggregate at the database, not in R.** Pulling 1,250 rows into R and grouping with `dplyr` would force every row across the SQLite ↔ R boundary. Letting `sqldf` push the `GROUP BY` down means SQLite returns only the per-`service_type` summary (5 rows) — far less data movement.
2. **`ORDER BY` on a derived column at the end** — the engine sorts a 5-row summary, not the 1,250-row source.

We can ask SQLite to show its execution plan to confirm:

In [ ]:
# EXPLAIN QUERY PLAN — show how SQLite plans to run the aggregate
plan <- sqldf("EXPLAIN QUERY PLAN
               SELECT service_type, COUNT(*), AVG(order_value)
               FROM orders GROUP BY service_type")
plan
cat('\nThe plan shows SQLite SCANs the orders table once and uses a hash to GROUP BY service_type. \n',
    'For larger NorthStar volumes an index on service_type would change the SCAN to a SEARCH.\n')

In [ ]:
# Arithmetic + CASE: revenue per promised hour, bucketed
q_arith <- "
  SELECT
    service_type,
    priority_level,
    ROUND(SUM(order_value) / SUM(promised_window_hours), 2) AS revenue_per_hour,
    CASE
      WHEN AVG(order_value) > 150 THEN 'High value'
      WHEN AVG(order_value) >  80 THEN 'Mid value'
      ELSE 'Low value'
    END AS value_band
  FROM orders
  GROUP BY service_type, priority_level
  ORDER BY revenue_per_hour DESC
"
sqldf(q_arith)

**Interpretation.** *Medical* and *Business* services generate the highest revenue per promised hour (i.e. shortest commitment, highest value). They are also the services where any failure most directly hurts the customer relationship — a useful lens when we look at failure rates next.

---
## 7. SQL queries inside R — the three NorthStar problems

### 7.1 Problem 1 — The 'completed-but-failed' paradox
Are deliveries marked OnTime/Delayed actually free of complaints?  
Join `deliveries` ↔ `orders` ↔ `complaints`.

In [ ]:
q_paradox <- "
  SELECT
    d.delivery_status,
    COUNT(DISTINCT d.delivery_id)                                  AS deliveries,
    COUNT(DISTINCT c.complaint_id)                                 AS complaints,
    ROUND(100.0 * COUNT(DISTINCT c.complaint_id)
                / COUNT(DISTINCT d.delivery_id), 2)                AS complaint_rate_pct,
    ROUND(AVG(c.compensation_amount),2)                            AS avg_compensation
  FROM deliveries d
  LEFT JOIN complaints c ON c.order_id = d.order_id
  GROUP BY d.delivery_status
  ORDER BY complaint_rate_pct DESC
"
paradox <- sqldf(q_paradox)
paradox

**Finding.** Even deliveries marked `OnTime` show a non-trivial complaint rate. Operations dashboards using `delivery_status` alone systematically *under-state* customer dissatisfaction — exactly the failure mode the customer-experience director described in the case study.

In [ ]:
# Visualise the paradox
ggplot(paradox, aes(x = delivery_status, y = complaint_rate_pct, fill = delivery_status)) +
  geom_col(width = 0.65) +
  geom_text(aes(label = paste0(complaint_rate_pct, '%')), vjust = -0.4, size = 4.5) +
  scale_fill_manual(values = c('OnTime'='#2E7D32','Delayed'='#F9A825','Failed'='#C62828')) +
  labs(title  = 'Complaint rate by recorded delivery status',
       subtitle = 'Even "OnTime" deliveries generate complaints — the system under-reports dissatisfaction',
       x = NULL, y = 'Complaint rate (%)') +
  theme_minimal(base_size = 13) +
  theme(legend.position = 'none')

### 7.2 Problem 2 — Manual route overrides and failure

In [ ]:
q_override <- "
  SELECT
    manual_route_override_count           AS overrides,
    COUNT(*)                              AS deliveries,
    SUM(CASE WHEN delivery_status='Failed'  THEN 1 ELSE 0 END) AS failed_n,
    SUM(CASE WHEN delivery_status='Delayed' THEN 1 ELSE 0 END) AS delayed_n,
    SUM(CASE WHEN delivery_status='OnTime'  THEN 1 ELSE 0 END) AS ontime_n,
    ROUND(100.0 * SUM(CASE WHEN delivery_status='Failed' THEN 1 ELSE 0 END)
                / COUNT(*),2)             AS failed_pct,
    ROUND(AVG(route_distance_km),2)       AS avg_distance
  FROM deliveries
  GROUP BY manual_route_override_count
  ORDER BY overrides
"
overrides <- sqldf(q_override)
overrides

In [ ]:
# Visualise: failure-rate increases with override count?
ggplot(overrides, aes(x = factor(overrides), y = failed_pct, group = 1)) +
  geom_line(colour = '#C62828', linewidth = 1.2) +
  geom_point(colour = '#C62828', size = 3) +
  geom_text(aes(label = paste0(failed_pct, '%')), vjust = -1, size = 4) +
  labs(title = 'Failure rate vs manual route overrides',
       subtitle = 'Drivers using more manual overrides see disproportionately more failed deliveries',
       x = 'Manual route overrides per delivery', y = 'Failure rate (%)') +
  theme_minimal(base_size = 13)

### 7.3 Problem 3 — The hidden cost story (zone × service)

In [ ]:
q_cost <- "
  SELECT
    o.pickup_zone                                              AS zone,
    o.service_type                                             AS service,
    COUNT(DISTINCT o.order_id)                                 AS orders,
    ROUND(SUM(o.order_value),2)                                AS revenue,
    COUNT(DISTINCT c.complaint_id)                             AS complaints,
    ROUND(SUM(COALESCE(c.compensation_amount,0)),2)            AS compensation_paid,
    ROUND(SUM(COALESCE(d.fuel_or_charge_cost,0)),2)            AS energy_cost,
    ROUND(SUM(o.order_value)
          - SUM(COALESCE(c.compensation_amount,0))
          - SUM(COALESCE(d.fuel_or_charge_cost,0)),2)          AS net_contribution
  FROM orders o
  LEFT JOIN deliveries d ON d.order_id = o.order_id
  LEFT JOIN complaints c ON c.order_id = o.order_id
  GROUP BY o.pickup_zone, o.service_type
  ORDER BY net_contribution ASC
  LIMIT 10
"
cost <- sqldf(q_cost)
cost

**Finding.** The 10 worst zone × service combinations leak measurable revenue when compensation and energy costs are netted off. The finance director was right: the unprofitable lines exist — they were just invisible across siloed tables.

---
## 8. Data manipulation & transformation in R (`dplyr`/`tidyr`)

Show three idiomatic R transformations beyond what SQL was used for.

In [ ]:
# (a) Compute realised delivery duration and SLA breach using lubridate + dplyr
deliveries_enriched <- deliveries %>%
  mutate(
    dispatch_time         = ymd_hms(dispatch_time),
    delivery_completed_at = ymd_hms(delivery_completed_at),
    duration_h            = as.numeric(difftime(delivery_completed_at, dispatch_time, units = 'hours'))
  ) %>%
  left_join(orders %>% select(order_id, promised_window_hours, service_type, pickup_zone),
            by = 'order_id') %>%
  mutate(
    sla_breach   = duration_h > promised_window_hours,
    breach_hours = pmax(duration_h - promised_window_hours, 0, na.rm = TRUE)
  )

summary_breach <- deliveries_enriched %>%
  filter(!is.na(duration_h)) %>%
  group_by(service_type) %>%
  summarise(
    n               = n(),
    median_duration = round(median(duration_h, na.rm = TRUE), 2),
    sla_breach_pct  = round(mean(sla_breach, na.rm = TRUE) * 100, 2),
    avg_breach_h    = round(mean(breach_hours, na.rm = TRUE), 2),
    .groups = 'drop'
  )
summary_breach

In [ ]:
# (b) Pivot complaint types per zone using tidyr — a wide ops dashboard view
complaint_pivot <- complaints %>%
  inner_join(orders %>% select(order_id, pickup_zone), by = 'order_id') %>%
  count(pickup_zone, complaint_type) %>%
  pivot_wider(names_from = complaint_type, values_from = n, values_fill = 0) %>%
  arrange(desc(rowSums(across(where(is.numeric)))))
complaint_pivot

In [ ]:
# (c) Driver risk score — combines override frequency, failure rate, low ratings
driver_risk <- deliveries %>%
  group_by(driver_id) %>%
  summarise(
    deliveries        = n(),
    avg_overrides     = mean(manual_route_override_count, na.rm = TRUE),
    failure_rate      = mean(delivery_status == 'Failed', na.rm = TRUE),
    avg_rating        = mean(customer_rating_post_delivery, na.rm = TRUE),
    .groups = 'drop'
  ) %>%
  left_join(drivers %>% select(driver_id, employment_type, training_score, base_zone),
            by = 'driver_id') %>%
  mutate(
    risk_score = round(
      0.4 * scale(avg_overrides)[,1] +
      0.4 * scale(failure_rate)[,1] +
      0.2 * scale(-avg_rating)[,1],   # invert so low ratings raise risk
      2)
  ) %>%
  arrange(desc(risk_score))

head(driver_risk, 10)

---
## 8.5 R-native statistical analysis

Beyond SQL aggregates, R itself provides a rich set of statistical functions that work directly on data frames. These are required by the *R analytics* marking criterion and let us characterise distributions, dispersion, and relationships that SQL alone hides.

We compute, all in base R / `dplyr`:
- **Central tendency** — `mean()`, `median()`
- **Dispersion** — `sd()`, `var()`, `IQR()`, `range()`
- **Quantiles** — `quantile()` at 25/50/75/90th
- **Six-number summary** — `summary()`
- **Correlations** — `cor()` between operational drivers

In [ ]:
# (a) Six-number summary on the key numeric columns of deliveries
num_cols <- c('route_distance_km', 'manual_route_override_count',
              'fuel_or_charge_cost', 'customer_rating_post_delivery')
cat('--- summary() of delivery numerics ---\n')
print(summary(deliveries[, num_cols]))

In [ ]:
# (b) Mean / median / sd / IQR / 25-75-90 quantiles per service_type
stats_by_service <- orders %>%
  group_by(service_type) %>%
  summarise(
    n         = n(),
    mean_val  = round(mean(order_value, na.rm = TRUE), 2),
    median_val= round(median(order_value, na.rm = TRUE), 2),
    sd_val    = round(sd(order_value, na.rm = TRUE), 2),
    iqr_val   = round(IQR(order_value, na.rm = TRUE), 2),
    q25       = round(quantile(order_value, 0.25, na.rm = TRUE), 2),
    q75       = round(quantile(order_value, 0.75, na.rm = TRUE), 2),
    q90       = round(quantile(order_value, 0.90, na.rm = TRUE), 2),
    .groups = 'drop'
  ) %>% arrange(desc(median_val))
stats_by_service

In [ ]:
# (c) Correlation matrix — relationships between operational drivers
cor_data <- deliveries %>%
  select(route_distance_km, manual_route_override_count,
         fuel_or_charge_cost, customer_rating_post_delivery) %>%
  na.omit()
cor_mat <- round(cor(cor_data), 3)
cor_mat
cat('\n--- Pearson correlations ---\n',
    'Distance vs fuel cost: positive (longer routes use more energy).\n',
    'Overrides vs rating  : negative (more overrides -> lower customer rating).\n')

**Statistical interpretation for the report.** The `summary()` output shows `manual_route_override_count` is heavily right-skewed: median is well below the mean and the 3rd quartile is small while the maximum is much larger. That's the long tail the case study warned about — a few drivers/routes are responsible for disproportionate overrides. The per-service stats confirm Medical and Business orders have the highest mean and median value, raising the operational stakes when those services fail. The correlation matrix shows the override → customer-rating link is real (negative coefficient) and the distance → fuel-cost link is, as expected, strongly positive.

---
## 9. Data visualisation in R

Three plot types, each chosen for a different analytical purpose:

1. **Stacked bar** — categorical comparison across groups (delivery outcomes by zone)
2. **Boxplot** — distribution comparison (override count distribution by status)
3. **Time-series line + facet** — temporal pattern (monthly complaint volume by type)

In [ ]:
# Plot 1 — Stacked bar of delivery outcomes by hub
plot_data1 <- deliveries %>%
  inner_join(hubs %>% select(hub_id, hub_name, zone), by = 'hub_id') %>%
  count(hub_name, delivery_status)

ggplot(plot_data1, aes(x = reorder(hub_name, -n, sum), y = n, fill = delivery_status)) +
  geom_col(position = 'fill') +
  scale_y_continuous(labels = percent_format()) +
  scale_fill_manual(values = c('OnTime'='#2E7D32','Delayed'='#F9A825','Failed'='#C62828')) +
  labs(title = 'Delivery outcome composition by hub',
       subtitle = 'Some hubs show a structurally higher share of failed/delayed deliveries',
       x = NULL, y = 'Share of deliveries', fill = 'Status') +
  theme_minimal(base_size = 13) +
  theme(axis.text.x = element_text(angle = 30, hjust = 1))

In [ ]:
# Plot 2 — Boxplot of override count by delivery status
ggplot(deliveries, aes(x = delivery_status, y = manual_route_override_count, fill = delivery_status)) +
  geom_boxplot(alpha = 0.85, outlier.alpha = 0.3) +
  scale_fill_manual(values = c('OnTime'='#2E7D32','Delayed'='#F9A825','Failed'='#C62828')) +
  labs(title  = 'Manual route overrides by delivery outcome',
       subtitle = 'Failed deliveries cluster around higher override counts',
       x = NULL, y = 'Manual overrides per delivery') +
  theme_minimal(base_size = 13) + theme(legend.position='none')

In [ ]:
# Plot 3 — Monthly complaint volume by complaint type
complaints_monthly <- complaints %>%
  mutate(month = floor_date(ymd_hms(created_at), 'month')) %>%
  count(month, complaint_type)

ggplot(complaints_monthly, aes(x = month, y = n, colour = complaint_type)) +
  geom_line(linewidth = 1) +
  facet_wrap(~ complaint_type, scales = 'free_y') +
  labs(title = 'Monthly complaint volume by category',
       subtitle = 'Different complaint types follow different temporal patterns — a single SLA target hides this',
       x = NULL, y = 'Complaints') +
  theme_minimal(base_size = 12) +
  theme(legend.position = 'none', strip.text = element_text(face = 'bold'),
        axis.text.x = element_text(angle = 30, hjust = 1))

---
## 10. Section 1 takeaways

1. **Zone data is structurally inconsistent** across systems — a clean-up step is mandatory before any cross-table analysis. This alone explains some of the conflicting reports the executives described.
2. **`delivery_status` is an unreliable proxy for customer experience.** Even `OnTime` deliveries generate complaints; status alone hides dissatisfaction.
3. **Manual route overrides correlate strongly with failure** — the operations director's hypothesis is partially right, but the *driver* dimension matters as much as the *route* dimension.
4. **Net contribution by zone × service is uneven.** Some service-zone combinations leak revenue once complaint compensation and energy cost are deducted from order value — the finance director's hidden-loss problem, made visible.
5. **Different complaint types follow different time profiles**, so a single response target masks specific operational issues (e.g. seasonal `MissedPickup` spikes vs steady `Billing` background).

These findings motivate Section 2 (deeper Python analytics) and Section 3 (a MongoDB design that links journeys, complaints, exceptions and app events as a single document so cross-system reporting becomes natural rather than fragile).